# Pronóstico de demanda eléctrica del SEIN (COES) · MIA-203
Compara líneas base (persistencia, estacional, programación del COES, Ridge) con MLP, DLinear, CNN, LSTM, GRU y Transformer para pronosticar las **24 h siguientes**, con partición por fechas (train ≤2021, val 2022, test 2023–2026).

**Cómo usarlo (3 pasos):** 1) Entorno de ejecución → GPU (T4). 2) Ejecuta todas las celdas. 3) Si Colab se corta, vuelve a ejecutar **todo**.

**Nunca parte de cero:** cada época guarda en Drive los pesos y el optimizador (`ckpt_<semilla>.pt`).
- Si se corta a mitad de un entrenamiento → continúa desde la última época guardada.
- Lo ya terminado se omite.
- Si quieres entrenar **más**, sube `EPOCAS` (p. ej. 30 → 50) y vuelve a ejecutar: **continúa desde lo aprendido** (no repite las primeras épocas) y conserva el mejor modelo según la validación.
- Solo se reinicia si cambias la configuración del modelo (ventana, residual, calendario, paso, lr, lote): en ese caso usa otro nombre de experimento.

Todo vive en la carpeta de Drive `23 PROYECTO FINAL/04 PROYECTO COES DEMANDA` (código, datos y `results_colab/`).

In [ ]:
from pathlib import Path
import subprocess, sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_ = Path("/content/drive/MyDrive/MIA07 REDES NEURONALES Y APRENDIZAJE PROFUNDO")
    PROYECTO = next((c for c in (RAIZ_ / "23 PROYECTO FINAL" / "02_Repositorio_Codigo", RAIZ_ / "23 PROYECTO FINAL" / "04 PROYECTO COES DEMANDA",
                                 RAIZ_ / "24 PREVIOS PROYECTO FINAL" / "04 PROYECTO COES DEMANDA") if (c / "src" / "exp_v2.py").is_file()),
                    RAIZ_ / "23 PROYECTO FINAL" / "02_Repositorio_Codigo")
except ImportError:                       # ejecución local
    PROYECTO = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if not (PROYECTO / "src" / "experimento.py").is_file():
    raise FileNotFoundError(f"Ajusta PROYECTO; no encuentro src/experimento.py en {PROYECTO}")
print("Proyecto:", PROYECTO)

In [ ]:
import importlib.util
if importlib.util.find_spec("holidays") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "holidays"], check=True)
import torch
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no hay (usa CPU, será lento)")

# Datos: si falta el CSV limpio, se descargan del portal del COES (~1 min)
if not (PROYECTO / "data" / "coes_demanda_limpio.csv").is_file():
    subprocess.run([sys.executable, str(PROYECTO / "coes_descarga.py"), "--salida", str(PROYECTO / "data")], check=True, cwd=PROYECTO)
else:
    print("Datos presentes:", (PROYECTO / "data" / "coes_demanda_limpio.csv").name)

## Configuración de la corrida
- `MODO = "rapido"`: prueba de humo (1 semilla, pocos datos, minutos). `MODO = "completo"`: 3 semillas y todos los modelos y ablaciones.
- Los resultados se guardan en `results_colab/` (no pisan los de la PC).

In [ ]:
MODO = "completo"            # "rapido" o "completo"
SEMILLAS = [17, 42, 73] if MODO == "completo" else [17]
PASO = 3 if MODO == "completo" else 24       # horas entre orígenes de entrenamiento
EPOCAS, PACIENCIA = (30, 6) if MODO == "completo" else (3, 2)   # sube EPOCAS para seguir entrenando desde lo aprendido
SALIDA = "results_colab" if MODO == "completo" else "results_colab_rapido"
HILOS = 2

import os

def ejecutar(*extra):
    cmd = [sys.executable, "-u", "src/experimento.py", "--resultados", SALIDA, "--hilos", str(HILOS), *extra]
    p = subprocess.Popen(cmd, cwd=PROYECTO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for linea in p.stdout:
        if "Warning" not in linea:
            print(linea, end="")
    if p.wait() != 0:
        raise RuntimeError("falló: " + " ".join(extra))
    os.sync()   # fuerza a Drive a escribir lo guardado

comun = ["--paso", str(PASO), "--epocas", str(EPOCAS), "--paciencia", str(PACIENCIA), "--semillas", *map(str, SEMILLAS)]
EXPERIMENTOS = [   # (nombre, argumentos)
    ("mlp_res",        ["--modelo", "mlp", "--residual"]),
    ("mlp_dir",        ["--modelo", "mlp"]),
    ("dlinear_res",    ["--modelo", "dlinear", "--residual"]),
    ("cnn_res",        ["--modelo", "cnn", "--residual"]),
    ("gru_res",        ["--modelo", "gru", "--residual"]),
    ("lstm_res",       ["--modelo", "lstm", "--residual"]),
    ("transformer_res",["--modelo", "transformer", "--residual"]),
    # ablaciones sobre GRU (cada una cambia UN factor)
    ("gru_dir",        ["--modelo", "gru"]),
    ("gru_res_sincal", ["--modelo", "gru", "--residual", "--sin_calendario"]),
    ("gru_res_L24",    ["--modelo", "gru", "--residual", "--L", "24"]),
    ("gru_res_L72",    ["--modelo", "gru", "--residual", "--L", "72"]),
    ("transformer_dir",["--modelo", "transformer"]),
]

## Líneas base (persistencia, estacional, programación del COES, Ridge)

In [ ]:
ejecutar("--modelo", "base", "--nombre", "base")

## Entrenamiento de las redes
Cada experimento imprime la pérdida por época. Si se corta, **vuelve a ejecutar esta celda**: reanuda desde el último checkpoint y omite lo ya terminado. Los pesos finales quedan en `results_colab/<experimento>/modelo_<semilla>.pt`.

In [ ]:
import time
t0 = time.time()
for nombre, args in EXPERIMENTOS:
    print()
    print(f"===== {nombre} =====")
    ejecutar(*args, "--nombre", nombre, *comun)
print()
print(f"Tiempo de esta sesión: {(time.time()-t0)/60:.1f} min")


## Resultados: tabla, comparaciones pareadas y figuras

In [ ]:
subprocess.run([sys.executable, "src/resumen.py", SALIDA], cwd=PROYECTO, check=True)
subprocess.run([sys.executable, "src/figuras.py", SALIDA], cwd=PROYECTO, check=True)
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv(PROYECTO / SALIDA / "tabla_principal.csv").round(2))
for f in ("fig_mae_modelos", "fig_mae_por_hora", "fig_mae_tipo_dia", "fig_ejemplo_2semanas"):
    display(Image(str(PROYECTO / SALIDA / f"{f}.png")))

## Lectura
Reporta en el artículo: MAE/MAPE/RMSE (media ± desv. de 3 semillas), la tabla de ablaciones (residual vs. directo, con/sin calendario, ventana de 24/72/168 h), el error por hora y por tipo de día, y el costo (columna `seg_train` y el dispositivo en `semilla_*.json`). Las comparaciones pareadas (`comparaciones_pareadas.csv`) traen un IC 95 % por bootstrap de bloques de 7 días.